# Test OpenRouter

This notebook shows:
- We can run a test case across multiple LLMs by changing model parameter in run_benchmark()
- GPT-OSS 20B via OpenRouter took 4.3 seconds and gave a clean answer (compared to 100s using Qwen 7B)

In [1]:
import sys
sys.path.insert(0, "../src")

from cx_agent.llm import chat
from cx_agent.data import load_fabsa
from cx_agent.tracing import run_with_trace
from cx_agent.evaluation import evaluate_record, run_benchmark, summarise

df_reviews, df_exploded = load_fabsa()

c:\Users\User\miniconda3\envs\cx-agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # test auto-router
# response = chat(
#     [{"role": "user", "content": "Reply with exactly: pong"}],
#     model="openrouter/openrouter/free",
# )
# print(response.choices[0].message.content)

In [4]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
# fetch list of free models with tool calling  

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
)
models = response.json()["data"]

free_tool_models = [
    m["id"] for m in models
    if m.get("pricing", {}).get("prompt") == "0"
    and "tools" in m.get("supported_parameters", [])
]

print(f"Free models with tool calling: {len(free_tool_models)}")
for m in free_tool_models:
    print(f"  openrouter/{m}")

Free models with tool calling: 18
  openrouter/nex-agi/nex-n2-pro:free
  openrouter/nvidia/nemotron-3-ultra-550b-a55b:free
  openrouter/openrouter/owl-alpha
  openrouter/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
  openrouter/poolside/laguna-xs.2:free
  openrouter/poolside/laguna-m.1:free
  openrouter/google/gemma-4-26b-a4b-it:free
  openrouter/google/gemma-4-31b-it:free
  openrouter/nvidia/nemotron-3-super-120b-a12b:free
  openrouter/openrouter/free
  openrouter/nvidia/nemotron-3-nano-30b-a3b:free
  openrouter/nvidia/nemotron-nano-12b-v2-vl:free
  openrouter/qwen/qwen3-next-80b-a3b-instruct:free
  openrouter/nvidia/nemotron-nano-9b-v2:free
  openrouter/openai/gpt-oss-120b:free
  openrouter/openai/gpt-oss-20b:free
  openrouter/qwen/qwen3-coder:free
  openrouter/meta-llama/llama-3.3-70b-instruct:free


In [ ]:
# test 1: GPT-OSS 20B (open weights)
record = run_with_trace(
    question="What are the top complaints in Banking?",
    df_reviews=df_reviews,
    df_exploded=df_exploded,
    model="openrouter/openai/gpt-oss-20b:free",
    expected_tool="describe",
    question_type="descriptive",
)

print(f"Answer: {record['answer']}")
print(f"Metrics: {evaluate_record(record)}")

Answer: The top complaints in Banking are app‑website (146 mentions), attitude‑of‑staff (109), ease‑of‑use (107), account‑access (102), and general‑satisfaction (76).
Metrics: {'num_tool_calls': 1, 'num_loops_detected': 0, 'tools_used': ['describe'], 'skill_correct': True, 'path_efficient': True, 'answer_has_markup': False, 'latency_seconds': 4.39}


In [9]:
# test 2: 1 question, 3 LLMs
test_case = [{
    "question": "What are the top complaints in Banking?",
    "expected_tool": "describe",
    "question_type": "descriptive",
}]

models = [
    "ollama/qwen2.5:7b-instruct",   # local baseline                                       
    "openrouter/nvidia/nemotron-nano-9b-v2:free",   # similar size to qwen 7B                        
    "openrouter/google/gemma-4-26b-a4b-it:free",    # different architecture                                            
]

for m in models:
    print(f"\n=== {m} ===")
    try:
        df = run_benchmark(test_case, df_reviews, df_exploded, model=m)
        print(f"Answer: {df.iloc[0]['answer']}")
        print(f"Tools used: {df.iloc[0]['tools_used']}")
        print(f"Latency: {df.iloc[0]['latency_seconds']}s")
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {str(e)[:200]}")


=== ollama/qwen2.5:7b-instruct ===
[1/1] What are the top complaints in Banking?...
Answer: Agent reached max steps. Last tool result: {"total_labels": 632, "unique_reviews": 318, "sentiment_distribution": {"negative": 1.0}, "top": [{"child_a
Tools used: ['describe']
Latency: 265.05s

=== openrouter/nvidia/nemotron-nano-9b-v2:free ===
[1/1] What are the top complaints in Banking?...

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Answer: The top complaints in Banking are app-website (146 mentions), attitude-of-staff (109), and ease-of-use (107).

Tools used: ['describe']
Latency: 10.87s

=== openrouter/google/gemma-4-26b-a4b-it:free ===
[1/1] What are the top complaints in Banking?...

Provider List: h